# 複数の特徴量からランキングを学習する（Learning to Rank、LambdaMART）

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_search/learning_to_rank.ipynb)

NLP クックブック 22-13 ／ 検索

BM25 のスコア、埋め込みの類似度、文書の長さなど、手元にある複数の手がかりをどう組み合わせて並べるかを、関連度のラベル付きデータから学習したい。クイズの質問と Wikipedia の段落に関連の有無のラベルを付けた JQaRA を使い、LightGBM の LambdaMART で学習して、BM25、埋め込み、RRF と比べる。

In [1]:
!pip install -q rank_bm25 fugashi unidic-lite

In [2]:
import numpy as np
import pandas as pd

# JQaRA: クイズの質問ごとに Wikipedia の段落の候補（dev は 50 件、test は 100 件）が付いている。
# ラベルは、答えの文字列を含み、答えを導くのに有用と確かめた段落が 1、それ以外が 0
base = "hf://datasets/hotchpotch/JQaRA/data/"
dev = pd.read_parquet(base + "dev-00000-of-00001.parquet")
test = pd.read_parquet(base + "test-00000-of-00001.parquet")
for name, d in [("dev", dev), ("test", test)]:
    by_pos = d.groupby(d.groupby("q_id").cumcount())["label"].mean()   # 配布時の並び順の i 番目が正解である割合
    print(f"{name}: {d['q_id'].nunique()} 質問, 正解の割合 (並び順の 1〜8 番目) {by_pos.head(8).round(2).tolist()}")

# 配布時の並び順は、dev でも test でも上位ほど正解が多い。並び順が答えを漏らさないよう、候補の順番は混ぜる
# 時間の都合で、dev（学習用）と test（評価用）から 100 質問ずつ選ぶ
rng = np.random.default_rng(0)
def sample(d, n):
    qids = rng.choice(d["q_id"].unique(), n, replace=False)
    return d[d["q_id"].isin(qids)].sample(frac=1, random_state=0).sort_values("q_id", kind="stable")

df = pd.concat([sample(dev, 100).assign(split="train"), sample(test, 100).assign(split="test")], ignore_index=True)
assert df.groupby("q_id")["split"].nunique().max() == 1      # 同じ質問が学習と評価の両方に入っていない
print(len(df), "組 |", df["question"].iloc[0], df["answers"].iloc[0])

dev: 1737 質問, 正解の割合 (並び順の 1〜8 番目) [1.0, 0.85, 0.67, 0.49, 0.29, 0.0, 0.0, 0.0]
test: 1667 質問, 正解の割合 (並び順の 1〜8 番目) [0.61, 0.45, 0.37, 0.33, 0.29, 0.24, 0.22, 0.22]
15000 組 | 『騎士団長殺し』『1Q84』『ノルウェイの森』といった小説の作者は誰でしょう? ['村上春樹']


In [3]:
from fugashi import Tagger
from rank_bm25 import BM25Okapi

tagger = Tagger()
def tokenize(text):
    return [w.feature.orthBase or w.surface for w in tagger(text.lower())
            if w.feature.pos1 not in ("助詞", "助動詞", "補助記号", "空白")]

titles = [tokenize(t) for t in df["title"]]
bodies = [tokenize(t) for t in df["text"]]
# IDF は手元の候補全体（200 質問分）から計算する簡略版。本来は検索対象の全文書から計算する
bm25 = BM25Okapi([t + b for t, b in zip(titles, bodies)])   # タイトル + 本文
bm25_title = BM25Okapi(titles)                               # タイトルだけ

rows = {}
for qid, g in df.groupby("q_id", sort=False):
    q = tokenize(g["question"].iloc[0]); qs = set(q); idx = g.index.to_list()
    for i, s, st in zip(idx, bm25.get_batch_scores(q, idx), bm25_title.get_batch_scores(q, idx)):
        rows[i] = {"bm25": s, "bm25_title": st,
                   "coverage": len(qs & (set(titles[i]) | set(bodies[i]))) / len(qs),   # クエリの語のうち文書に出てくる割合
                   "title_coverage": len(qs & set(titles[i])) / len(qs)}
df = df.join(pd.DataFrame.from_dict(rows, orient="index"))
df["doc_len"] = df["text"].str.len()

# 第 1 段: BM25 で各質問の上位 20 件に絞る。以降はこの 20 件の並べ替えを比べる
df["bm25_rank"] = df.groupby("q_id")["bm25"].rank(ascending=False, method="first")
cand = df[df["bm25_rank"] <= 20].reset_index(drop=True)
recall = cand.groupby("q_id")["label"].sum() / df.groupby("q_id")["label"].sum()
print(len(cand), "候補 | 正解のうち上位 20 件に残った割合:", recall.groupby(cand.groupby("q_id")["split"].first()).mean().round(3).to_dict())

4000 候補 | 正解のうち上位 20 件に残った割合: {'test': 0.515, 'train': 0.934}


In [4]:
from sentence_transformers import SentenceTransformer

# 4000 段落の埋め込み。Colab の CPU で 8〜12 分かかる
model = SentenceTransformer("intfloat/multilingual-e5-small")
P = model.encode(["passage: " + t + " " + x for t, x in zip(cand["title"], cand["text"])], batch_size=32, normalize_embeddings=True)
questions = cand["question"].unique().tolist()
Q = dict(zip(questions, model.encode(["query: " + q for q in questions], normalize_embeddings=True)))
cand["dense"] = (P * np.stack([Q[q] for q in cand["question"]])).sum(axis=1)   # コサイン類似度
cand["dense_rank"] = cand.groupby("q_id")["dense"].rank(ascending=False, method="first")
print(cand[["bm25", "bm25_title", "dense", "coverage", "title_coverage", "doc_len", "label"]].describe().round(2).loc[["mean", "min", "max"]])

        bm25  bm25_title  dense  coverage  title_coverage  doc_len  label
mean   30.28        5.04   0.84      0.35            0.05    252.9    0.2
min     1.97        0.00   0.76      0.05            0.00     11.0    0.0
max   117.46       40.13   0.94      0.92            0.46    400.0    1.0


In [5]:
import lightgbm as lgb

features = ["bm25", "bm25_title", "dense", "coverage", "title_coverage", "doc_len"]
params = dict(objective="lambdarank", n_estimators=100, learning_rate=0.05, num_leaves=7, min_child_samples=20, random_state=0, verbose=-1)
train, ev = cand[cand["split"] == "train"], cand[cand["split"] == "test"].copy()

# group: クエリごとの候補数を、行の並び順どおりに並べたもの。同じクエリの行は連続していなければならない
group = train.groupby("q_id", sort=False).size().to_numpy()
assert (train["q_id"] != train["q_id"].shift()).sum() == len(group)
print("group:", group[:8], "... 合計", group.sum(), "=", len(train), "行")

ranker = lgb.LGBMRanker(**params)
ranker.fit(train[features], train["label"], group=group)
ev["ltr"] = ranker.predict(ev[features])
ev["rrf"] = 1 / (60 + ev["bm25_rank"]) + 1 / (60 + ev["dense_rank"])    # この章の「キーワード検索と埋め込み検索を組み合わせる（ハイブリッド検索、RRF）」と同じ RRF (k=60)

group: [20 20 20 20 20 20 20 20] ... 合計 2000 = 2000 行


In [6]:
def reciprocal_rank(ranked, relevant):
    for i, doc in enumerate(ranked, 1):
        if doc in relevant:
            return 1 / i
    return 0.0

def ndcg_at_k(ranked, gains, k):
    dcg = sum(gains.get(doc, 0) / np.log2(i + 1) for i, doc in enumerate(ranked[:k], 1))
    ideal = sum(g / np.log2(i + 1) for i, g in enumerate(sorted(gains.values(), reverse=True)[:k], 1))
    return float(dcg / ideal) if ideal else 0.0

# 正解は第 1 段で落ちたものも含めて数える（理想の nDCG は元の候補全体の正解から計算）
gold = {qid: {d: 1 for d in s} for qid, s in df[df["label"] == 1].groupby("q_id")["id"]}

def evaluate(data, col):
    nd, rr = [], []
    for qid, g in data.groupby("q_id"):
        ranked = g.sort_values(col, ascending=False)["id"].tolist()
        nd.append(ndcg_at_k(ranked, gold[qid], 10)); rr.append(reciprocal_rank(ranked[:10], set(gold[qid])))
    return np.array(nd), np.array(rr)

methods = [("BM25", "bm25"), ("埋め込み", "dense"), ("RRF", "rrf"), ("LTR (LambdaMART)", "ltr"), ("上限 (20 件中の正解を全部上に)", "label")]
results = {name: evaluate(ev, col) for name, col in methods}
print(pd.DataFrame({k: {"nDCG@10": v[0].mean(), "MRR@10": v[1].mean()} for k, v in results.items()}).T.round(4))

def paired_ci(diff, n_boot=2000, seed=0):
    """質問ごとの差の平均の 95% 信頼区間（質問を復元抽出するブートストラップ）"""
    r = np.random.default_rng(seed)
    means = [diff[r.integers(0, len(diff), len(diff))].mean() for _ in range(n_boot)]
    return diff.mean(), *np.percentile(means, [2.5, 97.5])

for a, b in [("LTR (LambdaMART)", "RRF"), ("LTR (LambdaMART)", "BM25"), ("RRF", "埋め込み")]:
    diff = results[a][0] - results[b][0]
    m_, lo, hi = paired_ci(diff)
    print(f"nDCG@10 の差 {a} − {b}: {m_:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]"
          f"  (良い {(diff > 0).sum()} / 悪い {(diff < 0).sum()} / 同じ {(diff == 0).sum()} 質問)")

imp = pd.Series(ranker.booster_.feature_importance("gain"), index=features)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

                    nDCG@10  MRR@10
BM25                 0.4648  0.7397
埋め込み                 0.4613  0.6892
RRF                  0.4841  0.7408
LTR (LambdaMART)     0.4779  0.7275
上限 (20 件中の正解を全部上に)   0.7002  0.9200
nDCG@10 の差 LTR (LambdaMART) − RRF: -0.0062  95% CI [-0.0230, +0.0111]  (良い 37 / 悪い 43 / 同じ 20 質問)
nDCG@10 の差 LTR (LambdaMART) − BM25: +0.0132  95% CI [-0.0155, +0.0427]  (良い 53 / 悪い 33 / 同じ 14 質問)
nDCG@10 の差 RRF − 埋め込み: +0.0228  95% CI [+0.0028, +0.0429]  (良い 46 / 悪い 38 / 同じ 16 質問)
dense             0.889
bm25              0.052
coverage          0.023
doc_len           0.022
bm25_title        0.014
title_coverage    0.001
dtype: float64


In [7]:
# 比較: 評価と同じ分布のデータ（test の別の質問）で学習したらどうなるか。test の 100 質問を、質問単位で 5 分割して交差検証する
from sklearn.model_selection import GroupKFold

ev["ltr_cv"] = 0.0
imps = []
for tr_idx, te_idx in GroupKFold(n_splits=5).split(ev, groups=ev["q_id"]):
    a = ev.iloc[tr_idx]
    m = lgb.LGBMRanker(**params).fit(a[features], a["label"], group=a.groupby("q_id", sort=False).size().to_numpy())
    ev.iloc[te_idx, ev.columns.get_loc("ltr_cv")] = m.predict(ev.iloc[te_idx][features])
    imps.append(m.booster_.feature_importance("gain"))
nd, rr = evaluate(ev, "ltr_cv")
m_, lo, hi = paired_ci(nd - results["RRF"][0])
print(f"LTR (test の別の質問で学習): nDCG@10 {nd.mean():.4f}  MRR@10 {rr.mean():.4f}  | RRF との差 {m_:+.4f}  95% CI [{lo:+.4f}, {hi:+.4f}]")
imp = pd.Series(np.mean(imps, axis=0), index=features)
print((imp / imp.sum()).sort_values(ascending=False).round(3))

LTR (test の別の質問で学習): nDCG@10 0.4857  MRR@10 0.7498  | RRF との差 +0.0015  95% CI [-0.0201, +0.0231]
dense             0.402
bm25              0.179
coverage          0.164
doc_len           0.122
bm25_title        0.072
title_coverage    0.061
dtype: float64


- Learning to Rank (LTR) は、クエリと文書の組ごとの特徴量から並べる順番を学習する。文書ごとの関連度を当てる pointwise、同じクエリの 2 文書のどちらが上かを当てる pairwise、並び全体の良さを目的にする listwise に分けられる。LambdaMART は、ペアの順序の誤りに「入れ替えたら nDCG がどれだけ変わるか」で重みを付けた勾配を勾配ブースティング木で学習する方式で、LightGBM の objective="lambdarank" がこれにあたる。クロスエンコーダのスコア（この章の「クロスエンコーダで検索結果を並べ替える（リランキング）」）も特徴量の 1 つにできる。
- `group` は「最初の 20 行が 1 つ目のクエリ、次の 20 行が 2 つ目…」という件数の並びで、合計が行数と一致しなければならない。同じクエリの行は連続させる。ここでは全クエリが 20 件だが、件数はクエリごとに違ってよい。予測は行ごとのスコアなので `group` は要らず、クエリの中でスコア順に並べればよい。
- 学習と評価はクエリ単位で分ける。同じクエリの候補が両方に入ると、そのクエリに特有の傾向を覚えたまま評価することになり、数値が甘くなる。ここでは dev の質問で学習して test の質問で評価し（assert で確認）、最後のセルは `GroupKFold` で質問単位に分割している。また配布データの候補は上位ほど正解が多い（最初のセルの出力で、dev は 1〜5 番目が 1.0〜0.29 で 6 番目以降は 0、test も 1〜8 番目で 0.61 から 0.22 に下がる）ので、順番が答えを漏らさないよう混ぜてから使う。データの質問と答えは JAQKET 由来で CC BY-SA 4.0、段落は Wikipedia 由来（CC BY-SA 4.0 または GFDL）。
- 100 質問の評価では、どの方法の差も誤差の範囲にある。dev で学習した LTR は nDCG@10 0.478 で RRF の 0.484 より低いが、質問ごとの差のブートストラップ 95% 信頼区間は [−0.023, +0.011] で 0 をまたぎ、勝ち負けも 37 対 43 である。BM25 との差（+0.013、[−0.016, +0.043]）も同様。RRF と埋め込みの差（+0.023、[+0.003, +0.043]）だけは 0 をまたがなかったが、いくつも比べたうちの 1 つなので強くは言えない。信頼区間の考え方は評価指標と統計的検定の章の「ブートストラップ法で評価値の信頼区間を求める」を参照。LTR を入れるなら、もっと多くの質問で調整なしの RRF より良いことを確かめてからにする。
- dev と test では性質が違う。dev で学習したモデルの重要度は dense に偏り (0.889)、test の別の質問で学習すると bm25 や coverage にも分散する (dense 0.402)。BM25 の上位 20 件に残る正解の割合も dev 93.4%、test 51.5% と違う。README によれば、候補数が dev 50 件・test 100 件で、dev は正解を LLM で確かめた最大 5 件に絞っており、test のほうが正解が多い。また候補自体が e5-large など 5 つの埋め込みモデルの検索結果を RRF で絞ったもので、ここでの BM25 の第 1 段はその中での絞り込みにすぎない。dense の重要度が高いのは、候補を作った方法と同系統だからという可能性もある。ただし、これらが性能差の原因だとは、この実験からは言えない。なお最後のセルの LTR（nDCG@10 0.486、RRF との差 +0.002、[−0.020, +0.023]）は、評価と同じ分布のラベルで学習しているのに対し、BM25・埋め込み・RRF はラベルを使っていない。しかも特徴量や設定は、試行の段階で test の結果を見ながら決めたので、楽観的な参考値である。
- 第 1 段で落ちた正解は並べ替えでは取り戻せない。20 件の候補の中の正解を全部上に並べたときの上限は nDCG@10 0.700・MRR@10 0.920 で、どの方法 (0.46〜0.49) も方法同士の差よりはるかに大きく上限を下回る。test の正解のうち上位 20 件に残ったのは 51.5% なので、第 1 段をハイブリッド検索にする、候補を増やすなどで上限を上げる余地が大きい。
- 木のモデルは値の大小で分岐するので、特徴量の標準化は要らない。重要度 (gain) は分岐で損失を減らした量の合計で、似た特徴量の間で分け合われ、学習データで大きく変わる（上の 2 つのモデルのとおり）。ラベルは 0 以上の整数で、0〜3 のような段階もそのまま渡せる。LightGBM の lambdarank は既定で関連度 l の重みを 2^l − 1 とし、31 以上のラベルはエラーになる（`label_gain` で変えられる）。この章の「検索・ランキングを評価する（Recall@k、MRR、nDCG）」の `ndcg_at_k` は関連度をそのまま足すので、段階のあるラベルでは LightGBM の nDCG と値が一致しない（0/1 なら同じ）。
- クリックログを正解の代わりにすると、上位に表示された文書ほどクリックされる位置バイアスまで学習してしまう。クリックされなかった文書は、見られていないだけかもしれない。表示位置ごとのクリックされやすさを推定して補正する方法 (unbiased learning to rank) があり、LightGBM の `Dataset` には表示位置を渡す `position` 引数がある（4.6.0 で確認）。